# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

The dataset describes protein abundance, modifications, and sequences from human samples, captured using mass spectrometry analysis of extracellular vesicles.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print a summary from the metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs. The Croissant schema organizes tabular data as **record sets**. Each record set contains fields (columns) describing different attributes.

We enumerate all record sets (by their `@id`) and list their fields with their corresponding field `@id` and `name`. This overview helps select slices of data for extraction and analysis.

In [ ]:
# List all record sets and their fields by @id and name.

record_sets = dataset.metadata.recordSets
print("Available Record Sets (by `@id`):")
for rs in record_sets:
    print(f"  - {rs.id} : {rs.name}")

print("\nFields for Each Record Set:")
for rs in record_sets:
    print(f"\nRecord Set: {rs.name} (@id={rs.id})")
    for f in rs.fields:
        print(f"    Field @id: {f.id} | Name: {f.name} | DataType: {getattr(f, 'dataType', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, we demonstrate extraction for all record sets.

In [ ]:
# Extract all record sets into DataFrames, keyed by record set @id
dataframes = {}
for rs in dataset.metadata.recordSets:
    print(f"Extracting: {rs.name} (recordSet @id: {rs.id}) ...", flush=True)
    records = list(dataset.records(record_set=rs.id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"  -> Columns: {list(df.columns)} | Rows: {len(df)}")
    else:
        print("  -> [No records found]")

# Choose a main record set for further analysis (typically the largest/main table)
main_record_set_id = None
if dataframes:
    # Choose the one with most rows
    main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k]))
    print(f"\nUsing main record set for analysis: {main_record_set_id}")
    print("Sample data:")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets with data found.")

## 4. Exploratory Data Analysis (EDA)
Let's examine and process fields from the main record set identified above.

- We'll select a numeric field (e.g. protein abundance, counts, or molecular weight), filter based on value, normalize, and optionally group by a categorical variable if present.

In [ ]:
df = dataframes[main_record_set_id]

# Find numeric fields (float/int) by analyzing column types or by known field IDs/names.
# For this dataset, possible numeric fields are: 'MW', 'Abundance', 'Coverage', 'PeptideCount', etc.

import numpy as np

# Heuristic: choose the first float/integer-like column
numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) or np.issubdtype(df[col].dtype, np.number)]
# If missing dtype, try to coerce numeric
for col in df.columns:
    if col not in numeric_candidates:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except:
            pass
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_candidates.append(col)

if numeric_candidates:
    numeric_field = numeric_candidates[0] # Select first available numeric field
    print(f"Using numeric field: {numeric_field}")
else:
    print("No obvious numeric field found. Aborting EDA.")

# Filtering: Set a threshold
threshold = df[numeric_field].quantile(0.75) if numeric_candidates else None
if numeric_candidates and threshold is not None:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize this numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()

    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field (e.g. 'Sample', 'Condition', etc.)
    group_candidates = [col for col in df.select_dtypes(include=[object, 'category']).columns if col != numeric_field]
    group_field = group_candidates[0] if group_candidates else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")
else:
    print("No filtering or normalization performed due to missing numeric field.")

## 5. Visualization
Let's plot field distributions and relationships between variables in the main record set.

- Histogram of the selected numeric field.
- If a categorical field is present, boxplot or violinplot by category.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

if numeric_candidates:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We explored the FAIR² mass spectrometry dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library. We:
- Loaded dataset metadata and enumerated record sets and fields.
- Extracted tabular data using precise Croissant `@id` references.
- Performed basic EDA with field filtering, normalization, and grouping.
- Visualized numeric variable distribution and group effects.

For further analysis, consult the metadata for descriptions of fields, consider advanced visualizations, or integrate with downstream ML workflows.

For more information and dataset updates, visit: [https://sen.science/doi/10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa)